# Verificação Facial no LFW com YOLOv8 + AdaFace

Este notebook documenta o passo a passo completo do pipeline de **verificação facial** no dataset **LFW (Labeled Faces in the Wild)** usando dois modelos:

- **YOLOv8** — detecção e recorte de faces
- **AdaFace (IR-50)** — extração de embeddings faciais

## Estrutura do Projeto

```
deepfaketeste/
├── Dockerfile
├── docker-compose.gpu.yml
├── docker-compose.cpu.yml
├── requirements.txt
└── app/
    ├── net.py              # Arquitetura do AdaFace (IResNet Backbone)
    ├── baseline_lfw.py     # Script principal do baseline
    ├── download_lfw.py     # Script auxiliar para baixar o dataset
    └── test_gpu.py         # Script de teste da GPU
```

## Requisitos
- Windows com Docker Desktop instalado e em execução
- GPU NVIDIA com suporte CUDA (ex: RTX 4050)
- NVIDIA Container Toolkit configurado

---
## Parte 1 — Configuração do Ambiente Docker

Todo o projeto roda dentro de um container Docker com suporte a GPU NVIDIA.

### 1.1 Dockerfile

O Dockerfile usa a imagem oficial da NVIDIA com CUDA 12.2.2 + cuDNN 8 sobre Ubuntu 22.04.

> **Pontos importantes:**
> - `libgl1` e `libglib2.0-0` são necessários para o OpenCV funcionar dentro do container
> - `CMD ["bash"]` mantém o container rodando em modo interativo

```dockerfile
FROM nvidia/cuda:12.2.2-cudnn8-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive

WORKDIR /app

# Instalar Python 3.10 e dependências do sistema
RUN apt update && apt install -y \
    python3.10 \
    python3-pip \
    python3.10-dev \
    build-essential \
    git \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setar python padrão
RUN ln -s /usr/bin/python3.10 /usr/bin/python

COPY requirements.txt .

RUN pip install --upgrade pip
RUN pip install -r requirements.txt

COPY . .

CMD ["bash"]
```

### 1.2 docker-compose.gpu.yml

```yaml
services:
  ia:
    build: .
    container_name: ia_gpu
    volumes:
      - .:/app       # monta o projeto inteiro dentro do container
    deploy:
      resources:
        reservations:
          devices:
            - capabilities: [ gpu ]
    environment:
      - NVIDIA_VISIBLE_DEVICES=all
    tty: true
    stdin_open: true
```

### 1.3 requirements.txt

```
torch
torchvision
torchaudio
transformers
numpy
pandas
scikit-learn
ultralytics
opencv-python-headless
matplotlib
scipy
gdown
```

### 1.4 Comandos para subir o container

Execute os comandos abaixo no PowerShell **a partir da pasta do projeto**:

```powershell
# 1. Certificar que o Docker Desktop está aberto e rodando

# 2. Construir a imagem e subir o container em segundo plano
docker compose -f docker-compose.gpu.yml up --build -d

# 3. Verificar se o container está Up
docker compose -f docker-compose.gpu.yml ps

# 4. Entrar no container
docker exec -it ia_gpu bash
```

> **Dica:** Se aparecer erro de container com nome duplicado, remova o antigo:
> ```powershell
> docker rm -f ia_gpu
> ```

---
## Parte 2 — Teste da GPU

Antes de rodar o baseline, verifique se a GPU está acessível dentro do container.

Crie e execute o arquivo `app/test_gpu.py`:

In [ ]:
# app/test_gpu.py
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA disponível:", torch.cuda.is_available())
print("Device usado:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(3, 3).to(device)
print("Tensor está em:", x.device)
print(x)

Dentro do container, execute:

```bash
python app/test_gpu.py
```

Saída esperada (exemplo com RTX 4050):
```
CUDA disponível: True
Device usado: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Tensor está em: cuda:0
```

---
## Parte 3 — Dataset LFW

O **LFW (Labeled Faces in the Wild)** é o benchmark padrão para verificação facial.

- **13.233 imagens** de rostos de celebridades coletadas da web
- **5.749 identidades** diferentes
- Protocolo padrão: **6.000 pares** (3.000 positivos + 3.000 negativos) divididos em 10 folds

### 3.1 Download via scikit-learn (exploração inicial)

In [ ]:
# app/download_lfw.py
from sklearn.datasets import fetch_lfw_people
import os

output_dir = "data/lfw"
os.makedirs(output_dir, exist_ok=True)

# Baixa e carrega o LFW com pessoas que têm >= 1 foto
lfw = fetch_lfw_people(data_home=output_dir, download_if_missing=True)

print("Download concluído.")
print("Quantidade de imagens:", len(lfw.images))
print("Formato das imagens:", lfw.images.shape)
print("Quantidade de pessoas:", len(lfw.target_names))

Dentro do container:

```bash
python app/download_lfw.py
```

Saída esperada:
```
Download concluído.
Quantidade de imagens: 8579
Formato das imagens: (8579, 62, 47)
Quantidade de pessoas: 2526
```

> **Nota:** O `fetch_lfw_people` baixa o dataset LFW-funneled (imagens alinhadas). Para o pipeline completo com YOLO + AdaFace, o script principal baixa as imagens originais em alta resolução automaticamente.

---
## Parte 4 — Arquitetura: YOLOv8 + AdaFace

### Pipeline de Verificação Facial

```
Imagem A ─┐
           ├─► YOLOv8-face ─► Crop ─► AdaFace ─► Embedding A ─┐
Imagem B ─┘                                                     ├─► Similaridade Cosseno ─► Decisão
                               Crop ─► AdaFace ─► Embedding B ─┘
```

| Componente | Descrição |
|---|---|
| **YOLOv8n-face** | Detecta e localiza rostos na imagem com bounding box |
| **AdaFace IR-50** | Rede residual profunda que extrai embedding 512-d do rosto |
| **Cosine Similarity** | Mede similaridade entre dois embeddings (-1 a 1) |
| **Threshold** | Define o limiar de decisão: acima = mesma pessoa |

### Por que AdaFace?
O AdaFace adapta a função de perda de acordo com a **qualidade da imagem**: imagens de baixa qualidade recebem menor peso no treinamento. Isso o torna robusto para cenários do mundo real.

### 4.1 Arquitetura do AdaFace — net.py

O arquivo `app/net.py` implementa o **Backbone IResNet** do AdaFace, compatível com os checkpoints oficiais.

> **Atenção:** A ordem das camadas no bloco residual deve ser exatamente:
> `BN → Conv → BN → PReLU → Conv → BN`
> (diferente de implementações genéricas)

Estrutura geral do modelo:
- `input_layer`: Conv3×3 → BN → PReLU
- `body`: 24 blocos residuais (IR-50 = [3, 4, 14, 3] blocos por estágio)
- `output_layer`: BN → Dropout → Flatten → Linear(512×7×7 → 512) → BN

---
## Parte 5 — Script Principal: baseline_lfw.py

O script `app/baseline_lfw.py` implementa o pipeline completo.

### 5.1 Constantes e configurações

In [ ]:
import os
import sys

# Adicione o diretório app ao path se estiver rodando o notebook fora do container
sys.path.insert(0, 'app')

DATA_DIR = "data/lfw"
LFW_DIR = os.path.join(DATA_DIR, "lfw_home", "lfw_funneled")
PAIRS_FILE = os.path.join(DATA_DIR, "pairs.txt")
MODELS_DIR = "data/models"

# Google Drive ID do checkpoint AdaFace IR-50 (WebFace4M)
ADAFACE_GDRIVE_ID = "1HdW-F1GxJv0MVBUIVpE6HAZ3S9SLsytL"
ADAFACE_CKPT = os.path.join(MODELS_DIR, "adaface_ir50_webface4m.ckpt")

# YOLOv8-face weights
YOLO_FACE_URL = "https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov8n-face.pt"
YOLO_FACE_PATH = os.path.join(MODELS_DIR, "yolov8n-face.pt")

FACE_SIZE = (112, 112)  # tamanho exigido pelo AdaFace

### 5.2 Download automático dos modelos

In [ ]:
import urllib.request
import tarfile

def download_file(url_or_id, dest, is_gdrive=False):
    """Faz download de uma URL ou de um arquivo do Google Drive."""
    if os.path.exists(dest):
        print(f"Já existe: {dest}")
        return
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if is_gdrive:
        import gdown
        print(f"Baixando do Google Drive...")
        gdown.download(id=url_or_id, output=dest, quiet=False)
    else:
        print(f"Baixando: {url_or_id}")
        urllib.request.urlretrieve(url_or_id, dest)
        print(f"Salvo em: {dest}")

def ensure_lfw_images():
    """Verifica ou baixa as imagens LFW-funneled."""
    if os.path.isdir(LFW_DIR) and len(os.listdir(LFW_DIR)) > 100:
        print(f"Imagens LFW encontradas em: {LFW_DIR}")
        return
    tgz_path = os.path.join(DATA_DIR, "lfw-funneled.tgz")
    lfw_home = os.path.join(DATA_DIR, "lfw_home")
    os.makedirs(lfw_home, exist_ok=True)
    download_file("http://vis-www.cs.umass.edu/lfw/lfw-funneled.tgz", tgz_path)
    print("Extraindo...")
    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall(path=lfw_home)
    print("Extração concluída!")

# Download do pairs.txt (protocolo oficial LFW)
PAIRS_URL = "https://raw.githubusercontent.com/davidsandberg/facenet/master/data/pairs.txt"

# Executar downloads
ensure_lfw_images()
download_file(PAIRS_URL, PAIRS_FILE)
download_file(ADAFACE_GDRIVE_ID, ADAFACE_CKPT, is_gdrive=True)
download_file(YOLO_FACE_URL, YOLO_FACE_PATH)

### 5.3 Carregar os modelos

In [ ]:
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from net import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── YOLOv8-face ──────────────────────────────────
print("\nCarregando YOLOv8-face...")
yolo = YOLO(YOLO_FACE_PATH)

# ── AdaFace IR-50 ────────────────────────────────
print("Carregando AdaFace IR-50...")

def load_adaface(checkpoint_path, architecture="ir_50"):
    """Carrega o modelo AdaFace com pesos pré-treinados."""
    model = build_model(architecture, mode='ir')  # mode='ir' para o checkpoint WebFace4M
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    statedict = checkpoint["state_dict"]
    # O checkpoint oficial usa prefixo "model." — removê-lo
    model_statedict = {
        key[len("model."):]: val
        for key, val in statedict.items()
        if key.startswith("model.")
    }
    model.load_state_dict(model_statedict)
    model.eval()
    return model

adaface = load_adaface(ADAFACE_CKPT).to(DEVICE)
print("Modelos carregados com sucesso!")

### 5.4 Detecção de face com YOLOv8

In [ ]:
import cv2
import numpy as np

def detect_and_crop_face(yolo_model, img_bgr, target_size=(112, 112)):
    """
    Detecta a face com maior confiança na imagem usando YOLOv8,
    recorta com margem e redimensiona para o tamanho do AdaFace.
    Se nenhuma face for detectada, usa a imagem inteira como fallback.
    """
    results = yolo_model(img_bgr, verbose=False)
    best_box, best_conf = None, 0.0

    for result in results:
        if result.boxes is not None and len(result.boxes) > 0:
            for i, conf in enumerate(result.boxes.conf):
                if conf.item() > best_conf:
                    best_conf = conf.item()
                    best_box = result.boxes.xyxy[i].cpu().numpy()

    if best_box is not None:
        x1, y1, x2, y2 = best_box.astype(int)
        h_img, w_img = img_bgr.shape[:2]
        bw, bh = x2 - x1, y2 - y1
        # Adiciona 15% de margem ao redor da face
        mx, my = int(bw * 0.15), int(bh * 0.15)
        x1, y1 = max(0, x1 - mx), max(0, y1 - my)
        x2, y2 = min(w_img, x2 + mx), min(h_img, y2 + my)
        face = img_bgr[y1:y2, x1:x2]
    else:
        # Fallback: LFW já é centrado em rostos
        face = img_bgr

    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face_resized = cv2.resize(face_rgb, target_size)
    return face_resized

### 5.5 Extração de embeddings com AdaFace

In [ ]:
def preprocess_face(face_rgb):
    """
    Normaliza uma face RGB (np array HxWx3) para tensor AdaFace.
    Pixels normalizados para [-1, 1] e reorganizados para CHW.
    """
    face = cv2.resize(face_rgb, FACE_SIZE).astype(np.float32) / 255.0
    face = (face - 0.5) / 0.5  # normalização para [-1, 1]
    face = np.transpose(face, (2, 0, 1))  # HWC → CHW
    return torch.FloatTensor(face).unsqueeze(0)  # adiciona dimensão de batch


@torch.no_grad()
def get_embedding(model, face_tensor, device):
    """
    Extrai o embedding L2-normalizado de uma face.
    O AdaFace Backbone retorna (embedding, norm).
    """
    face_tensor = face_tensor.to(device)
    output = model(face_tensor)
    # Backbone retorna tupla (embedding_normalizado, norm)
    embedding = output[0] if isinstance(output, tuple) else output
    embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy().flatten()

### 5.6 Parser do protocolo LFW (pairs.txt)

In [ ]:
def parse_pairs(pairs_file, lfw_dir):
    """
    Lê o pairs.txt do protocolo oficial LFW.
    Retorna lista de (caminho_img1, caminho_img2, é_mesma_pessoa).
    
    Formato do pairs.txt:
    - 1ª linha: N_folds\tN_pares
    - Pares positivos: nome\tindice1\tindice2  (mesma pessoa)
    - Pares negativos: nome1\tindice1\tnome2\tindice2  (pessoas diferentes)
    """
    pairs = []
    with open(pairs_file, "r") as f:
        lines = f.readlines()

    header = lines[0].strip().split("\t")
    n_folds = int(header[0])
    n_pairs = int(header[1])
    idx = 1

    for fold in range(n_folds):
        # Pares positivos
        for _ in range(n_pairs):
            parts = lines[idx].strip().split("\t")
            name = parts[0]
            i1, i2 = int(parts[1]), int(parts[2])
            p1 = os.path.join(lfw_dir, name, f"{name}_{i1:04d}.jpg")
            p2 = os.path.join(lfw_dir, name, f"{name}_{i2:04d}.jpg")
            pairs.append((p1, p2, True))
            idx += 1

        # Pares negativos
        for _ in range(n_pairs):
            parts = lines[idx].strip().split("\t")
            n1, i1 = parts[0], int(parts[1])
            n2, i2 = parts[2], int(parts[3])
            p1 = os.path.join(lfw_dir, n1, f"{n1}_{i1:04d}.jpg")
            p2 = os.path.join(lfw_dir, n2, f"{n2}_{i2:04d}.jpg")
            pairs.append((p1, p2, False))
            idx += 1

    return pairs

pairs = parse_pairs(PAIRS_FILE, LFW_DIR)
n_pos = sum(1 for _, _, s in pairs if s)
n_neg = len(pairs) - n_pos
print(f"Total de pares: {len(pairs)}")
print(f"  Positivos (mesma pessoa): {n_pos}")
print(f"  Negativos (pessoas diferentes): {n_neg}")

### 5.7 Processar os pares

In [ ]:
import time

similarities, labels, errors = [], [], 0
t0 = time.time()

for i, (p1, p2, is_same) in enumerate(pairs):
    # Verificar se as imagens existem
    if not os.path.exists(p1) or not os.path.exists(p2):
        errors += 1
        continue

    # Carregar imagens
    img1, img2 = cv2.imread(p1), cv2.imread(p2)
    if img1 is None or img2 is None:
        errors += 1
        continue

    # Detectar e recortar faces com YOLOv8
    face1 = detect_and_crop_face(yolo, img1, FACE_SIZE)
    face2 = detect_and_crop_face(yolo, img2, FACE_SIZE)

    # Extrair embeddings com AdaFace
    emb1 = get_embedding(adaface, preprocess_face(face1), DEVICE)
    emb2 = get_embedding(adaface, preprocess_face(face2), DEVICE)

    # Calcular similaridade cosseno
    sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-8)
    similarities.append(sim)
    labels.append(1 if is_same else 0)

    if (i + 1) % 1000 == 0:
        el = time.time() - t0
        eta = el / (i + 1) * (len(pairs) - i - 1)
        print(f"{i+1}/{len(pairs)} | {el:.0f}s decorridos | ETA: {eta:.0f}s")

total_time = time.time() - t0
print(f"\nConcluído em {total_time:.1f}s")
if errors:
    print(f"⚠ {errors} pares ignorados (imagens não encontradas)")

### 5.8 Avaliação e Métricas

In [ ]:
from sklearn.metrics import roc_curve, auc, accuracy_score

sims = np.array(similarities)
labs = np.array(labels)

# ── Melhor threshold por grid search ──────────────
best_acc, best_t = 0, 0
for t in np.arange(0.0, 1.0, 0.001):
    preds = (sims >= t).astype(int)
    acc = accuracy_score(labs, preds)
    if acc > best_acc:
        best_acc, best_t = acc, t

# ── AUC e EER ─────────────────────────────────────
fpr, tpr, _ = roc_curve(labs, sims)
roc_auc = auc(fpr, tpr)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fpr - fnr))]

print("="*50)
print("RESULTADOS — LFW Face Verification")
print("="*50)
print(f"Pares avaliados : {len(labs)}")
print(f"Melhor threshold: {best_t:.4f}")
print(f"Acurácia        : {best_acc*100:.2f}%")
print(f"AUC             : {roc_auc:.4f}")
print(f"EER             : {eer*100:.2f}%")
print(f"Tempo total     : {total_time:.1f}s")

### 5.9 Acurácia por fold (10-fold cross-validation)

In [ ]:
n_folds = 10
fold_size = len(labs) // n_folds
fold_accs = []

print("Acurácia por fold:")
for f in range(n_folds):
    s, e = f * fold_size, (f + 1) * fold_size
    preds_fold = (sims[s:e] >= best_t).astype(int)
    acc_fold = accuracy_score(labs[s:e], preds_fold)
    fold_accs.append(acc_fold)
    print(f"  Fold {f+1:2d}: {acc_fold*100:.2f}%")

print(f"  {'─'*30}")
print(f"  Média : {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%")

### 5.10 Plot da curva ROC

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2,
         label=f'AdaFace IR-50 (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.scatter([eer], [1 - eer], color='red', zorder=5,
            label=f'EER = {eer*100:.2f}%')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Curva ROC — LFW Face Verification\nYOLOv8 + AdaFace IR-50')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/roc_curve_lfw.png', dpi=150)
plt.show()
print("Curva ROC salva em data/roc_curve_lfw.png")

---
## Resumo do Pipeline

```
1. Docker (NVIDIA CUDA 12.2.2 + cuDNN8 + Ubuntu 22.04)
   └─ Python 3.10 + PyTorch + Ultralytics + OpenCV

2. Dataset LFW
   ├─ 13.233 imagens, 5.749 identidades
   └─ Protocolo: 6.000 pares (10 folds × 600 pares)

3. Detecção de Face
   └─ YOLOv8n-face → bounding box + crop 112×112

4. Embedding Facial
   └─ AdaFace IR-50 (WebFace4M) → vetor 512-d normalizado

5. Verificação
   └─ Similaridade Cosseno → threshold → Mesma/Diferente pessoa

6. Métricas
   ├─ Acurácia por threshold ótimo
   ├─ AUC (Area Under ROC Curve)
   └─ EER (Equal Error Rate)
```

## Para executar o script completo

```bash
# Dentro do container:
python app/baseline_lfw.py
```